In [2]:
#qt for gui figures, inline for embedded in notebook 
%matplotlib qt
import hyperspy.api as hs
import exspy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd



In [3]:
s = hs.load("JS006_0p31.rpl", stack=True)
s.set_signal_type("EDS_SEM")
s.metadata.General.title = 'Mulit-phase Jadeite Region'
s


  0%|          | 0/1 [00:00<?, ?it/s]

<EDSSEMSpectrum, title: Mulit-phase Jadeite Region, dimensions: (1024, 768|2665)>

In [4]:
s.axes_manager.gui() #enter pixel and -0.96 offset keV and 0.01 keV, scale is 0.31 micron

In [5]:
s = s.rebin((1024/8,768/8,2665))


In [6]:
s_cropped = s.isig[0.2:10.0]

In [7]:
#PCA/ICA
s_cropped.change_dtype('float')
s_cropped.decomposition(normalize_poissonian_noise=True)

/Users/thomaslam/anaconda3/lib/python3.13/site-packages/rsciio/utils/rgb_tools.py:62: VisibleDeprecationWarning: The module `rsciio.utils.rgb_tools` has been renamed to `rsciio.utils.rgb` and it will be removed in version 1.0. Use `RGB_DTYPES` instead
  warnings.warn(
/Users/thomaslam/anaconda3/lib/python3.13/site-packages/rsciio/utils/rgb_tools.py:62: VisibleDeprecationWarning: The module `rsciio.utils.rgb_tools` has been renamed to `rsciio.utils.rgb` and it will be removed in version 1.0.
  warnings.warn(


Decomposition info:
  normalize_poissonian_noise=True
  algorithm=SVD
  output_dimension=None
  centre=None


In [8]:
s_cropped.plot_explained_variance_ratio()

<Axes: title={'center': 'Mulit-phase Jadeite Region\nPCA Scree Plot'}, xlabel='Principal component index', ylabel='Proportion of variance'>

In [9]:
s_cropped.change_dtype('float')
s_cropped.decomposition(algorithm='NMF', solver='mu', output_dimension=5, normalize_poissonian_noise=True, random_state=42)

Decomposition info:
  normalize_poissonian_noise=True
  algorithm=NMF
  output_dimension=5
  centre=None
scikit-learn estimator:
NMF(n_components=5, random_state=42, solver='mu')


In [10]:
s_cropped.plot_decomposition_results(True)

/Users/thomaslam/anaconda3/lib/python3.13/site-packages/rsciio/utils/rgb_tools.py:62: VisibleDeprecationWarning: The module `rsciio.utils.rgb_tools` has been renamed to `rsciio.utils.rgb` and it will be removed in version 1.0.
  warnings.warn(


/Users/thomaslam/anaconda3/lib/python3.13/site-packages/rsciio/utils/rgb_tools.py:62: VisibleDeprecationWarning: The module `rsciio.utils.rgb_tools` has been renamed to `rsciio.utils.rgb` and it will be removed in version 1.0.
  warnings.warn(
/Users/thomaslam/anaconda3/lib/python3.13/site-packages/rsciio/utils/rgb_tools.py:62: VisibleDeprecationWarning: The module `rsciio.utils.rgb_tools` has been renamed to `rsciio.utils.rgb` and it will be removed in version 1.0.
  warnings.warn(


In [11]:
from pathlib import Path

loadings = s_cropped.learning_results.loadings
height, width = 96, 128

outdir = Path("output")
outdir.mkdir(exist_ok=True)

for i in range(5):
    loading_map = loadings[:, i].reshape(height, width)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(loading_map, cmap="gray")
    ax.axis("off")

    fig.savefig(
        outdir / f"nmf_component_{i}_gray.png",
        dpi=150,
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.close(fig)

    print(f"Saved nmf_component_{i}_gray.png")

Saved nmf_component_0_gray.png
Saved nmf_component_1_gray.png
Saved nmf_component_2_gray.png
Saved nmf_component_3_gray.png
Saved nmf_component_4_gray.png


In [12]:
from skimage.filters import threshold_otsu
from skimage import io
from skimage import io, transform

# Load component 0 
img_rgba = io.imread('output/nmf_component_0_gray.png')

# Convert RGBA to grayscale (take just the first channel or luminance)
if len(img_rgba.shape) == 3 and img_rgba.shape[2] == 4:
    # Take first channel (grayscale saved as RGBA)
    img = img_rgba[:, :, 0].astype(np.float32)
    print(f"Converted RGBA to grayscale. Shape: {img.shape}")
else:
    img = img_rgba.astype(np.float32)

# Resize to 96 x 128
img_96x128 = transform.resize(
    img,
    (96, 128),
    order=1,
    preserve_range=True,
    anti_aliasing=True
).astype(np.float32)

print(f"Resized image shape: {img_96x128.shape}")

# Apply Otsu thresholding
threshold = threshold_otsu(img_96x128)
binary = img_96x128 >= threshold

# Get area fraction and save as variable C0_AFrac, where C0 is Component 0
C0_AFrac = binary.sum() / binary.size

# Save the binary mask as PNG
outdir = Path("output")
mask_uint8 = (binary * 255).astype(np.uint8)
mask_fname = outdir / "nmf_component_0_mask.png"
io.imsave(mask_fname, mask_uint8, check_contrast=False)
print(f"Saved mask: {mask_fname}")

# Create figure with 3 panels
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Resized grayscale image
axes[0].imshow(img_96x128, cmap='gray')
axes[0].set_title(f'Component 0\nShape: {img_96x128.shape}')
axes[0].axis('off')

# Panel 2: Histogram with threshold line
axes[1].hist(img_96x128.flatten(), bins=256, color='gray', alpha=0.7)
axes[1].axvline(threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Threshold = {threshold:.2f}')
axes[1].set_title(f'Histogram with Otsu Threshold\nArea Fraction: {C0_AFrac:.2%}')
axes[1].set_xlabel('Pixel Intensity')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Panel 3: Binary mask (thresholded)
axes[2].imshow(binary, cmap='gray')
axes[2].set_title(f'Binary Mask\nForeground: {binary.sum()} pixels\nBackground: {(~binary).sum()} pixels')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"  Image shape: {img_96x128.shape}")
print(f"  Otsu threshold: {threshold:.2f}")
print(f"  C0_AFrac = {C0_AFrac:.2%}")
print(f"  Foreground pixels: {binary.sum()}")
print(f"  Background pixels: {(~binary).sum()}")
print(f"  Total pixels: {binary.size}")

Converted RGBA to grayscale. Shape: (871, 1162)
Resized image shape: (96, 128)
Saved mask: output/nmf_component_0_mask.png
  Image shape: (96, 128)
  Otsu threshold: 171.34
  C0_AFrac = 89.06%
  Foreground pixels: 10944
  Background pixels: 1344
  Total pixels: 12288


In [13]:
# Load component 1
img_rgba = io.imread('output/nmf_component_1_gray.png')

# Convert RGBA to grayscale (take just the first channel or luminance)
if len(img_rgba.shape) == 3 and img_rgba.shape[2] == 4:
    # Take first channel (grayscale saved as RGBA)
    img = img_rgba[:, :, 0].astype(np.float32)
    print(f"Converted RGBA to grayscale. Shape: {img.shape}")
else:
    img = img_rgba.astype(np.float32)

# Resize to 96 x 128
img_96x128 = transform.resize(
    img,
    (96, 128),
    order=1,
    preserve_range=True,
    anti_aliasing=True
).astype(np.float32)

print(f"Resized image shape: {img_96x128.shape}")

# Apply Otsu thresholding
threshold = threshold_otsu(img_96x128)
binary = img_96x128 >= threshold

# Get area fraction and save as variable C1_AFrac, where C1 is Component 1
C1_AFrac = binary.sum() / binary.size

# Save the binary mask as PNG
outdir = Path("output")
mask_uint8 = (binary * 255).astype(np.uint8)
mask_fname = outdir / "nmf_component_1_mask.png"
io.imsave(mask_fname, mask_uint8, check_contrast=False)
print(f"Saved mask: {mask_fname}")

# Create figure with 3 panels
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Resized grayscale image
axes[0].imshow(img_96x128, cmap='gray')
axes[0].set_title(f'Component 1\nShape: {img_96x128.shape}')
axes[0].axis('off')

# Panel 2: Histogram with threshold line
axes[1].hist(img_96x128.flatten(), bins=256, color='gray', alpha=0.7)
axes[1].axvline(threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Threshold = {threshold:.2f}')
axes[1].set_title(f'Histogram with Otsu Threshold\nArea Fraction: {C1_AFrac:.2%}')
axes[1].set_xlabel('Pixel Intensity')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Panel 3: Binary mask (thresholded)
axes[2].imshow(binary, cmap='gray')
axes[2].set_title(f'Binary Mask\nForeground: {binary.sum()} pixels\nBackground: {(~binary).sum()} pixels')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"  Image shape: {img_96x128.shape}")
print(f"  Otsu threshold: {threshold:.2f}")
print(f"  C1_AFrac = {C1_AFrac:.2%}")
print(f"  Foreground pixels: {binary.sum()}")
print(f"  Background pixels: {(~binary).sum()}")
print(f"  Total pixels: {binary.size}")

Converted RGBA to grayscale. Shape: (871, 1162)
Resized image shape: (96, 128)
Saved mask: output/nmf_component_1_mask.png
  Image shape: (96, 128)
  Otsu threshold: 80.16
  C1_AFrac = 9.24%
  Foreground pixels: 1136
  Background pixels: 11152
  Total pixels: 12288


In [14]:
# Load component 2
img_rgba = io.imread('output/nmf_component_2_gray.png')

# Convert RGBA to grayscale (take just the first channel or luminance)
if len(img_rgba.shape) == 3 and img_rgba.shape[2] == 4:
    # Take first channel (grayscale saved as RGBA)
    img = img_rgba[:, :, 0].astype(np.float32)
    print(f"Converted RGBA to grayscale. Shape: {img.shape}")
else:
    img = img_rgba.astype(np.float32)

# Resize to 96 x 128
img_96x128 = transform.resize(
    img,
    (96, 128),
    order=1,
    preserve_range=True,
    anti_aliasing=True
).astype(np.float32)

print(f"Resized image shape: {img_96x128.shape}")

# Apply Otsu thresholding
threshold = threshold_otsu(img_96x128)
binary = img_96x128 >= threshold

# Get area fraction and save as variable C2_AFrac, where C2 is Component 2
C2_AFrac = binary.sum() / binary.size

# Save the binary mask as PNG
outdir = Path("output")
mask_uint8 = (binary * 255).astype(np.uint8)
mask_fname = outdir / "nmf_component_2_mask.png"
io.imsave(mask_fname, mask_uint8, check_contrast=False)
print(f"Saved mask: {mask_fname}")

# Create figure with 3 panels
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Resized grayscale image
axes[0].imshow(img_96x128, cmap='gray')
axes[0].set_title(f'Component 2\nShape: {img_96x128.shape}')
axes[0].axis('off')

# Panel 2: Histogram with threshold line
axes[1].hist(img_96x128.flatten(), bins=256, color='gray', alpha=0.7)
axes[1].axvline(threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Threshold = {threshold:.2f}')
axes[1].set_title(f'Histogram with Otsu Threshold\nArea Fraction: {C2_AFrac:.2%}')
axes[1].set_xlabel('Pixel Intensity')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Panel 3: Binary mask (thresholded)
axes[2].imshow(binary, cmap='gray')
axes[2].set_title(f'Binary Mask\nForeground: {binary.sum()} pixels\nBackground: {(~binary).sum()} pixels')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"  Image shape: {img_96x128.shape}")
print(f"  Otsu threshold: {threshold:.2f}")
print(f"  C2_AFrac = {C2_AFrac:.2%}")
print(f"  Foreground pixels: {binary.sum()}")
print(f"  Background pixels: {(~binary).sum()}")
print(f"  Total pixels: {binary.size}")

Converted RGBA to grayscale. Shape: (871, 1162)
Resized image shape: (96, 128)
Saved mask: output/nmf_component_2_mask.png
  Image shape: (96, 128)
  Otsu threshold: 59.95
  C2_AFrac = 1.60%
  Foreground pixels: 196
  Background pixels: 12092
  Total pixels: 12288


In [15]:
# Load component 3
img_rgba = io.imread('output/nmf_component_3_gray.png')

# Convert RGBA to grayscale (take just the first channel or luminance)
if len(img_rgba.shape) == 3 and img_rgba.shape[2] == 4:
    # Take first channel (grayscale saved as RGBA)
    img = img_rgba[:, :, 0].astype(np.float32)
    print(f"Converted RGBA to grayscale. Shape: {img.shape}")
else:
    img = img_rgba.astype(np.float32)

# Resize to 96 x 128
img_96x128 = transform.resize(
    img,
    (96, 128),
    order=1,
    preserve_range=True,
    anti_aliasing=True
).astype(np.float32)

print(f"Resized image shape: {img_96x128.shape}")

# Apply Otsu thresholding
threshold = threshold_otsu(img_96x128)
binary = img_96x128 >= threshold

# Get area fraction and save as variable C3_AFrac, where C3 is Component 3
C3_AFrac = binary.sum() / binary.size

# Save the binary mask as PNG
outdir = Path("output")
mask_uint8 = (binary * 255).astype(np.uint8)
mask_fname = outdir / "nmf_component_3_mask.png"
io.imsave(mask_fname, mask_uint8, check_contrast=False)
print(f"Saved mask: {mask_fname}")

# Create figure with 3 panels
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Resized grayscale image
axes[0].imshow(img_96x128, cmap='gray')
axes[0].set_title(f'Component 3\nShape: {img_96x128.shape}')
axes[0].axis('off')

# Panel 2: Histogram with threshold line
axes[1].hist(img_96x128.flatten(), bins=256, color='gray', alpha=0.7)
axes[1].axvline(threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Threshold = {threshold:.2f}')
axes[1].set_title(f'Histogram with Otsu Threshold\nArea Fraction: {C3_AFrac:.2%}')
axes[1].set_xlabel('Pixel Intensity')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Panel 3: Binary mask (thresholded)
axes[2].imshow(binary, cmap='gray')
axes[2].set_title(f'Binary Mask\nForeground: {binary.sum()} pixels\nBackground: {(~binary).sum()} pixels')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"  Image shape: {img_96x128.shape}")
print(f"  Otsu threshold: {threshold:.2f}")
print(f"  C3_AFrac = {C3_AFrac:.2%}")
print(f"  Foreground pixels: {binary.sum()}")
print(f"  Background pixels: {(~binary).sum()}")
print(f"  Total pixels: {binary.size}")

Converted RGBA to grayscale. Shape: (871, 1162)
Resized image shape: (96, 128)
Saved mask: output/nmf_component_3_mask.png
  Image shape: (96, 128)
  Otsu threshold: 21.87
  C3_AFrac = 17.86%
  Foreground pixels: 2195
  Background pixels: 10093
  Total pixels: 12288


In [16]:
# Load masks
c1 = io.imread("output/nmf_component_1_mask.png")
c3 = io.imread("output/nmf_component_3_mask.png")

# Defining masks for white foreground / black background:
c1_fg = c1 > 128
c3_fg = c3 > 128

# Shared foreground
shared = c1_fg & c3_fg

# Make shared area black in Component 3
c3_new = c3.copy()
c3_new[shared] = 0

# Save result
outpath = Path("output/nmf_component_3_mask_blackshared.png")
io.imsave(outpath, c3_new.astype(np.uint8), check_contrast=False)

print(f"Saved: {outpath}")
print(f"Shared pixels made black: {shared.sum()}")

Saved: output/nmf_component_3_mask_blackshared.png
Shared pixels made black: 847


In [17]:
from skimage import io, morphology

# Denoise Component 3
c3 = io.imread("output/nmf_component_3_mask_blackshared.png")

# white = foreground
c3_fg = c3 > 128

# stronger cleanup
c3_clean = morphology.remove_small_objects(c3_fg, min_size=50, connectivity=2)
c3_clean = morphology.remove_small_holes(c3_clean, area_threshold=50)

# save back with white foreground
c3_out = np.where(c3_clean, 255, 0).astype(np.uint8)
io.imsave(Path("output/nmf_component_3_mask_clean.png"), c3_out, check_contrast=False)

print(f"Foreground pixels before: {c3_fg.sum()}")
print(f"Foreground pixels after:  {c3_clean.sum()}")

Foreground pixels before: 1348
Foreground pixels after:  617


In [18]:
# Corrected component 3 area fraction calc
img_rgba = io.imread('output/nmf_component_3_mask_clean.png')

# Convert RGBA to grayscale (take just the first channel or luminance)
if len(img_rgba.shape) == 3 and img_rgba.shape[2] == 4:
    # Take first channel (grayscale saved as RGBA)
    img = img_rgba[:, :, 0].astype(np.float32)
    print(f"Converted RGBA to grayscale. Shape: {img.shape}")
else:
    img = img_rgba.astype(np.float32)

# Apply Otsu thresholding
threshold = threshold_otsu(img)
binary = img >= threshold
# Get area fraction and save as variable C_C3_AFrac, where C_C3 is the Corrected Component 3
C_C3_AFrac = binary.sum() / binary.size

print(f"  C_C3_AFrac = {C_C3_AFrac:.2%}")
print(f"  Foreground pixels: {binary.sum()}")
print(f"  Background pixels: {(~binary).sum()}")
print(f"  Total pixels: {binary.size}")

  C_C3_AFrac = 5.02%
  Foreground pixels: 617
  Background pixels: 11671
  Total pixels: 12288


In [19]:
#C_C0 is the Corrected Componet 0
C_C0_AFrac = C0_AFrac - C_C3_AFrac
print(f" C_C0_AFrac = {C_C0_AFrac:.2%}")

 C_C0_AFrac = 84.04%


In [20]:
# Applying C1 mask to original data cube prior to NMF 

cube = s.rebin((1024/8, 768/8, 2665)).data

mask = io.imread("output/nmf_component_1_mask.png")
keep = mask > 128

cube1 = cube.reshape(-1, cube.shape[-1])
keep1 = keep.reshape(-1)

sum_spec_array = cube1[keep1].sum(axis=0)

# energy axis from original signal
x = s.axes_manager.signal_axes[0].axis

plt.figure(figsize=(8, 4))
plt.plot(x, sum_spec_array, lw=1)
plt.xlim(0.2, 7)
plt.xlabel(f"{s.axes_manager.signal_axes[0].name} ({s.axes_manager.signal_axes[0].units})")
plt.ylabel("Counts")
plt.title("Component 1 summed spectrum")

for val, label in [(1.04, "Na Ka"), (1.25, "Mg Ka"), (1.49, "Al Ka"), (1.74, "Si Ka"), (3.69, "Ca Ka"), (6.40, "Fe Ka")]:
    plt.axvline(val, color="r", ls="--", alpha=0.35)
    plt.text(val, plt.ylim()[1]*0.9, label, rotation=90, va="top", ha="right", fontsize=8)

plt.tight_layout()
plt.show()

In [21]:
# Quantification for Component 1

# Na background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 0.9) & (x <= 0.92)
right_bg = (x >= 1.15) & (x <= 1.18)
peak = (x >= 0.94) & (x <= 1.14)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Na_net = y_corr[peak].sum()
print("Na net counts", f"{Na_net:.3f}")

# Na2O Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Na_stds = pd.to_numeric(df.iloc[0, 1], errors="raise")
K_ratio_Na = Na_net/(net_counts_Na_stds * C1_AFrac)
print ("K_ratio_Na", f"{K_ratio_Na:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_Na2O = pd.to_numeric(df.iloc[2, 1], errors="raise")
mass_percent_Na2O = K_ratio_Na * Lookup_Na2O
print("Mass_percent_Na2O",f"{mass_percent_Na2O:.3f}")

# Mg background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 1.14) & (x <= 1.16)
right_bg = (x >= 1.36) & (x <= 1.38)
peak = (x >= 1.18) & (x <= 1.33)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Mg_net = y_corr[peak].sum()
print("Mg_net_counts", f"{Mg_net:.3f}")

# MgO Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Mg_stds = pd.to_numeric(df.iloc[0, 2], errors="raise")
K_ratio_Mg = Mg_net/(net_counts_Mg_stds * C1_AFrac)
print ("K_ratio_Mg", f"{K_ratio_Mg:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_MgO = pd.to_numeric(df.iloc[2, 2], errors="raise")
mass_percent_MgO = K_ratio_Mg * Lookup_MgO
print("Mass_percent_MgO",f"{mass_percent_MgO:.3f}")

# Al background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 1.35) & (x <= 1.4)
right_bg = (x >= 1.57) & (x <= 1.63)
peak = (x >= 1.41) & (x <= 1.56)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Al_net = y_corr[peak].sum()
print("Al_net_counts", f"{Al_net:.3f}")

# Al2O3 Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Al_stds = pd.to_numeric(df.iloc[0, 3], errors="raise")
K_ratio_Al = Al_net/(net_counts_Al_stds * C1_AFrac)
print ("K_ratio_Al", f"{K_ratio_Al:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_Al2O3 = pd.to_numeric(df.iloc[2, 3], errors="raise")
mass_percent_Al2O3 = K_ratio_Al * Lookup_Al2O3
print("Mass_percent_Al2O3",f"{mass_percent_Al2O3:.3f}")

# Si background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg  = (x >= 1.63) & (x <= 1.64)
right_bg = (x >= 1.93) & (x <= 1.94)
peak     = (x >= 1.64) & (x <= 1.93)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Si_net = y_corr[peak].sum()
print("Si_net_counts", f"{Si_net:.3f}")

# SiO2 Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Si_stds = pd.to_numeric(df.iloc[0, 4], errors="raise")
K_ratio_Si = Si_net/(net_counts_Si_stds * C1_AFrac)
print ("K_ratio_Si", f"{K_ratio_Si:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_SiO2 = pd.to_numeric(df.iloc[2, 4], errors="raise")
mass_percent_SiO2 = K_ratio_Si * Lookup_SiO2
print("Mass_percent_SiO2", f"{mass_percent_SiO2:.3f}")

# Ca background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 3.44) & (x <= 3.5)
right_bg = (x >= 3.82) & (x <= 3.88)
peak = (x >= 3.59) & (x <= 3.82)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Ca_net = y_corr[peak].sum()
print("Ca_net_counts", f"{Ca_net:.3f}")

# CaO Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Ca_stds = pd.to_numeric(df.iloc[0, 8], errors="raise")
K_ratio_Ca = Ca_net/(net_counts_Ca_stds * C1_AFrac)
print ("K_ratio_Ca", f"{K_ratio_Ca:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_CaO = pd.to_numeric(df.iloc[2, 8], errors="raise")
mass_percent_CaO = K_ratio_Ca * Lookup_CaO
print("Mass_percent_CaO", f"{mass_percent_CaO:.3f}")

# Fe background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 6.1) & (x <= 6.25)
right_bg = (x >= 6.7) & (x <= 6.75)
peak = (x >= 6.21) & (x <= 6.6)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Fe_net = y_corr[peak].sum()
print("Fe_net_counts", f"{Fe_net:.3f}")

# FeO Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Fe_stds = pd.to_numeric(df.iloc[0, 11], errors="raise")
K_ratio_Fe = Fe_net/(net_counts_Fe_stds * C1_AFrac)
print ("K_ratio_Fe", f"{K_ratio_Fe:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_FeO = pd.to_numeric(df.iloc[2, 11], errors="raise")
mass_percent_FeO = K_ratio_Fe * Lookup_FeO
print("Mass_percent_FeO", f"{mass_percent_FeO:.3f}")

Na net counts 53861.969
K_ratio_Na 0.766
Mass_percent_Na2O 8.913
Mg_net_counts 34463.596
K_ratio_Mg 0.110
Mass_percent_MgO 5.614
Al_net_counts 78107.565
K_ratio_Al 0.815
Mass_percent_Al2O3 13.644
Si_net_counts 277933.950
K_ratio_Si 0.867
Mass_percent_SiO2 56.179
Ca_net_counts 41799.346
K_ratio_Ca 0.201
Mass_percent_CaO 9.687
Fe_net_counts 5095.598
K_ratio_Fe 0.128
Mass_percent_FeO 2.991


In [22]:
# Export of Quant for component 1

out = pd.DataFrame([
    ["Na", K_ratio_Na, mass_percent_Na2O],
    ["Mg", K_ratio_Mg, mass_percent_MgO],
    ["Al", K_ratio_Al, mass_percent_Al2O3],
    ["Si", K_ratio_Si, mass_percent_SiO2],
    ["Ca", K_ratio_Ca, mass_percent_CaO],
    ["Fe", K_ratio_Fe, mass_percent_FeO]
], columns=["Element", "K_ratio", "Oxide_quant_pct"])

out = out.round(2)
out.to_csv("C1_Quant.csv", index=False)

In [23]:
# Applying Corrected C3 mask to original data cube prior to NMF 
cube = s.rebin((1024/8, 768/8, 2665)).data

# Corrected component 3
mask = io.imread("output/nmf_component_3_mask_clean.png")
keep = mask > 128

cube3 = cube.reshape(-1, cube.shape[-1])
keep3 = keep.reshape(-1)

sum_spec_array = cube3[keep3].sum(axis=0)

# energy axis from original signal
x = s.axes_manager.signal_axes[0].axis

plt.figure(figsize=(8, 4))
plt.plot(x, sum_spec_array, lw=1)
plt.xlim(0.2, 7)
plt.xlabel(f"{s.axes_manager.signal_axes[0].name} ({s.axes_manager.signal_axes[0].units})")
plt.ylabel("Counts")
plt.title("C_Component 3 summed spectrum")

for val, label in [(1.04, "Na Ka"), (1.49, "Al Ka"), (1.74, "Si Ka")]:
    plt.axvline(val, color="r", ls="--", alpha=0.35)
    plt.text(val, plt.ylim()[1]*0.9, label, rotation=90, va="top", ha="right", fontsize=8)

plt.tight_layout()
plt.show()

In [24]:
# C_Quantification for Corrected Component 3

# Na background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 0.84) & (x <= 0.90)
right_bg = (x >= 1.16) & (x <= 1.22)
peak = (x >= 0.96) & (x <= 1.12)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Na_net = y_corr[peak].sum()
print("Na net counts", f"{Na_net:.3f}")

# Na2O Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Na_stds = pd.to_numeric(df.iloc[0, 1], errors="raise")
K_ratio_Na = Na_net/(net_counts_Na_stds*C_C3_AFrac)
print ("K_ratio_Na", f"{K_ratio_Na:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_Na2O = pd.to_numeric(df.iloc[2, 1], errors="raise")
mass_percent_Na2O = K_ratio_Na * Lookup_Na2O
print("Mass_percent_Na2O",f"{mass_percent_Na2O:.3f}")

# Al background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 1.32) & (x <= 1.36)
right_bg = (x >= 1.58) & (x <= 1.62)
peak = (x >= 1.4) & (x <= 1.55)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Al_net = y_corr[peak].sum()
print("Al_net_counts", f"{Al_net:.3f}")

# Al2O3
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Al_stds = pd.to_numeric(df.iloc[0, 3], errors="raise")
K_ratio_Al = Al_net/(net_counts_Al_stds*C_C3_AFrac)
print ("K_ratio_Al", f"{K_ratio_Al:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_Al2O3 = pd.to_numeric(df.iloc[2, 3], errors="raise")
mass_percent_Al2O3 = K_ratio_Al * Lookup_Al2O3
print("Mass_percent_Al2O3",f"{mass_percent_Al2O3:.3f}")

# Si background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg  = (x >= 1.63) & (x <= 1.64)
right_bg = (x >= 1.93) & (x <= 1.94)
peak     = (x >= 1.64) & (x <= 1.93)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Si_net = y_corr[peak].sum()
print("Si_net_counts", f"{Si_net:.3f}")

# SiO2 Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Si_stds = pd.to_numeric(df.iloc[0, 4], errors="raise")
K_ratio_Si = Si_net/(net_counts_Si_stds*C_C3_AFrac)
print ("K_ratio_Si", f"{K_ratio_Si:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_SiO2 = pd.to_numeric(df.iloc[2, 4], errors="raise")
mass_percent_SiO2 = K_ratio_Si * Lookup_SiO2
print("Mass_percent_SiO2", f"{mass_percent_SiO2:.3f}")

#K background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 3.0) & (x <= 3.1)
right_bg = (x >= 3.48) & (x <= 3.52)
peak = (x >= 3.2) & (x <= 3.42)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
K_net = y_corr[peak].sum()
print("K_net_counts", f"{K_net:.3f}")

#K2O Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_K_stds = pd.to_numeric(df.iloc[0, 7], errors="raise")
K_ratio_K = K_net/(net_counts_K_stds * C_C3_AFrac)
print ("K_ratio_K", f"{K_ratio_K:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_K2O = pd.to_numeric(df.iloc[2, 7], errors="raise")
mass_percent_K2O = K_ratio_K * Lookup_K2O
print("mass_percent_K2O", f"{mass_percent_K2O:.3f}")

Na net counts 41312.133
K_ratio_Na 1.082
Mass_percent_Na2O 12.586
Al_net_counts 63115.500
K_ratio_Al 1.213
Mass_percent_Al2O3 20.300
Si_net_counts 174990.712
K_ratio_Si 1.005
Mass_percent_SiO2 65.124
K_net_counts 818.220
K_ratio_K 0.019
mass_percent_K2O 0.287


In [25]:
# Export of Quant for component 3 (corrected)

out = pd.DataFrame([
    ["Na", K_ratio_Na, mass_percent_Na2O],
    ["Al", K_ratio_Al, mass_percent_Al2O3],
    ["Si", K_ratio_Si, mass_percent_SiO2],
    ["K", K_ratio_K, mass_percent_K2O],
], columns=["Element", "K_ratio", "Oxide_quant_pct"])

out = out.round(2)
out.to_csv("C_C3_Quant.csv", index=False)

In [26]:
from PIL import Image, ImageOps

c3 = Image.open("output/nmf_component_3_mask_clean.png").convert("L")
c0 = Image.open("output/nmf_component_0_mask.png").convert("L")

c3_inv = ImageOps.invert(c3)
c3_inv.save("output/c3_inv.png")

a3 = np.array(c3_inv)
a0 = np.array(c0)

combined = np.minimum(a0, a3)

result = Image.fromarray(combined.astype(np.uint8))
result.save("output/C_C0_combined.png")




In [27]:
from PIL import Image, ImageOps

c3 = Image.open("output/nmf_component_3_mask_clean.png").convert("L")
c0 = Image.open("output/nmf_component_0_mask.png").convert("L")

c3_inv = ImageOps.invert(c3)
c3_inv.save("output/c3_inv.png")

a3 = np.array(c3_inv)
a0 = np.array(c0)

combined = np.minimum(a0, a3)

result = Image.fromarray(combined.astype(np.uint8))
result.save("output/C_C0_combined.png")




In [28]:
# Applying Corrected C0 mask to original data cube prior to NMF 
cube = s.rebin((1024/8, 768/8, 2665)).data

# Corrected component 3
mask = io.imread("output/C_C0_combined.png")
keep = mask > 128

cube3 = cube.reshape(-1, cube.shape[-1])
keep3 = keep.reshape(-1)

sum_spec_array = cube3[keep3].sum(axis=0)

# energy axis from original signal
x = s.axes_manager.signal_axes[0].axis

plt.figure(figsize=(8, 4))
plt.plot(x, sum_spec_array, lw=1)
plt.xlim(0.2, 7)
plt.xlabel(f"{s.axes_manager.signal_axes[0].name} ({s.axes_manager.signal_axes[0].units})")
plt.ylabel("Counts")
plt.title("C_Component 0 summed spectrum")

for val, label in [(1.04, "Na Ka"), (1.25, "Mg Ka"), (1.49, "Al Ka"), (1.74, "Si Ka"), (3.69, "Ca Ka")]:
    plt.axvline(val, color="r", ls="--", alpha=0.35)
    plt.text(val, plt.ylim()[1]*0.9, label, rotation=90, va="top", ha="right", fontsize=8)

plt.tight_layout()
plt.show()

In [29]:
# Quantification for Corrected Component 0

# Na background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 0.84) & (x <= 0.90)
right_bg = (x >= 1.16) & (x <= 1.22)
peak = (x >= 0.96) & (x <= 1.12)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Na_net = y_corr[peak].sum()
print("Na net counts", f"{Na_net:.3f}")

# Na2O Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Na_stds = pd.to_numeric(df.iloc[0, 1], errors="raise")
K_ratio_Na = Na_net/(net_counts_Na_stds * C_C0_AFrac)
print ("K_ratio_Na", f"{K_ratio_Na:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_Na2O = pd.to_numeric(df.iloc[2, 1], errors="raise")
mass_percent_Na2O = K_ratio_Na * Lookup_Na2O
print("Mass_percent_Na2O",f"{mass_percent_Na2O:.3f}")

# Mg background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 1.14) & (x <= 1.16)
right_bg = (x >= 1.36) & (x <= 1.38)
peak = (x >= 1.18) & (x <= 1.33)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Mg_net = y_corr[peak].sum()
print("Mg_net_counts", f"{Mg_net:.3f}")

# MgO Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Mg_stds = pd.to_numeric(df.iloc[0, 2], errors="raise")
K_ratio_Mg = Mg_net/(net_counts_Mg_stds * C_C0_AFrac)
print ("K_ratio_Mg", f"{K_ratio_Mg:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_MgO = pd.to_numeric(df.iloc[2, 2], errors="raise")
mass_percent_MgO = K_ratio_Mg * Lookup_MgO
print("Mass_percent_MgO",f"{mass_percent_MgO:.3f}")

# Al background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 1.32) & (x <= 1.36)
right_bg = (x >= 1.58) & (x <= 1.62)
peak = (x >= 1.4) & (x <= 1.55)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Al_net = y_corr[peak].sum()
print("Al_net_counts", f"{Al_net:.3f}")

# Al2O3 Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Al_stds = pd.to_numeric(df.iloc[0, 3], errors="raise")
K_ratio_Al = Al_net/(net_counts_Al_stds * C_C0_AFrac)
print ("K_ratio_Al", f"{K_ratio_Al:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_Al2O3 = pd.to_numeric(df.iloc[2, 3], errors="raise")
mass_percent_Al2O3 = K_ratio_Al * Lookup_Al2O3
print("Mass_percent_Al2O3",f"{mass_percent_Al2O3:.3f}")

# Si background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg  = (x >= 1.63) & (x <= 1.64)
right_bg = (x >= 1.93) & (x <= 1.94)
peak     = (x >= 1.64) & (x <= 1.93)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Si_net = y_corr[peak].sum()
print("Si_net_counts", f"{Si_net:.3f}")

# SiO2 Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Si_stds = pd.to_numeric(df.iloc[0, 4], errors="raise")
K_ratio_Si = Si_net/(net_counts_Si_stds * C_C0_AFrac)
print ("K_ratio_Si", f"{K_ratio_Si:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_SiO2 = pd.to_numeric(df.iloc[2, 4], errors="raise")
mass_percent_SiO2 = K_ratio_Si * Lookup_SiO2
print("Mass_percent_SiO2", f"{mass_percent_SiO2:.3f}")

#Ca background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 3.44) & (x <= 3.5)
right_bg = (x >= 3.82) & (x <= 3.88)
peak = (x >= 3.59) & (x <= 3.82)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Ca_net = y_corr[peak].sum()
print("Ca_net_counts", f"{Ca_net:.3f}")

#CaO Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Ca_stds = pd.to_numeric(df.iloc[0, 8], errors="raise")
K_ratio_Ca = Ca_net/(net_counts_Ca_stds * C_C0_AFrac)
print ("K_ratio_Ca", f"{K_ratio_Ca:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_CaO = pd.to_numeric(df.iloc[2, 8], errors="raise")
mass_percent_CaO = K_ratio_Ca * Lookup_CaO
print("Mass_percent_CaO", f"{mass_percent_CaO:.3f}")

#Fe background and net counts calc
x = s.axes_manager.signal_axes[0].axis
y = sum_spec_array

left_bg = (x >= 6.1) & (x <= 6.25)
right_bg = (x >= 6.7) & (x <= 6.75)
peak = (x >= 6.21) & (x <= 6.6)

x_bg = np.r_[x[left_bg], x[right_bg]]
y_bg = np.r_[y[left_bg], y[right_bg]]

coef = np.polyfit(x_bg, y_bg, 1)
baseline = np.polyval(coef, x)

y_corr = y - baseline
Fe_net = y_corr[peak].sum()
print("Fe_net_counts", f"{Fe_net:.3f}")

#FeO Quant
df = pd.read_csv("15kV_Stds_Lib.csv")
net_counts_Fe_stds = pd.to_numeric(df.iloc[0, 11], errors="raise")
K_ratio_Fe = Fe_net/(net_counts_Fe_stds * C_C0_AFrac)
print ("K_ratio_Fe", f"{K_ratio_Fe:.3f}")

df = pd.read_csv("15kV_Stds_Lib.csv")
Lookup_FeO = pd.to_numeric(df.iloc[2, 11], errors="raise")
mass_percent_FeO = K_ratio_Fe * Lookup_FeO
print("Mass_percent_FeO", f"{mass_percent_FeO:.3f}")

Na net counts 837629.978
K_ratio_Na 1.311
Mass_percent_Na2O 15.247
Mg_net_counts 23015.170
K_ratio_Mg 0.008
Mass_percent_MgO 0.412
Al_net_counts 1213803.500
K_ratio_Al 1.393
Mass_percent_Al2O3 23.324
Si_net_counts 2565436.205
K_ratio_Si 0.880
Mass_percent_SiO2 57.042
Ca_net_counts 33329.424
K_ratio_Ca 0.018
Mass_percent_CaO 0.850
Fe_net_counts 9850.189
K_ratio_Fe 0.027
Mass_percent_FeO 0.636


In [30]:
# Export of Quant for component 0 (corrected)

out = pd.DataFrame([
    ["Na", K_ratio_Na, mass_percent_Na2O],
    ["Mg", K_ratio_Mg, mass_percent_MgO],
    ["Al", K_ratio_Al, mass_percent_Al2O3],
    ["Si", K_ratio_Si, mass_percent_SiO2],
    ["Ca", K_ratio_Ca, mass_percent_CaO],
    ["Fe", K_ratio_Fe, mass_percent_FeO]
], columns=["Element", "K_ratio", "Oxide_quant_pct"])

out = out.round(2)
out.to_csv("C_C0_Quant.csv", index=False)

In [31]:
#Phase-specific summary

C_C0 = pd.read_csv("C_C0_Quant.csv")[["Element", "K_ratio", "Oxide_quant_pct"]]
C1 = pd.read_csv("C1_Quant.csv")[["Element", "K_ratio", "Oxide_quant_pct"]]
C_C3 = pd.read_csv("C_C3_Quant.csv")[["Element", "K_ratio", "Oxide_quant_pct"]]

C_C0 = C_C0.rename(columns={
    "K_ratio": "K_ratio_C_C0",
    "Oxide_quant_pct": "Oxide_quant_pct_Na"
})
C1 = C1.rename(columns={
    "K_ratio": "K_ratio_C1",
    "Oxide_quant_pct": "Oxide_quant_pct_Mg"
})
C_C3 = C_C3.rename(columns={
    "K_ratio": "K_ratio_C_C3",
    "Oxide_quant_pct": "Oxide_quant_pct_Al"
})

summary = C_C0.merge(C1, on="Element", how="outer").merge(C_C3, on="Element", how="outer")
summary.to_csv("Component_Summary.csv", index=False)